# Decode the Bioreactor — 02 Mechanistic model
Simulation per batch: start from the day-0 sample → integrate the bucket equations (forward Euler, dt = 0.005 d) → at each feed apply the mixing jump (volume up, dilute, add feed contents) → continue to the end.
Parameters are **global** (shared by all batches), fitted by least squares. A batch is predicted from its day-0 values + feed schedule only.

Model and equations: `src/bioreactor/model.py`. Fit: `src/bioreactor/fit.py`.

**Runtime**: one simulation of ~40 batches ≈ 0.1–0.3 s; a fit needs many simulations (17 parameters × finite differences), so a fit takes minutes. Start with section 3 (one fold, few iterations).

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from bioreactor.data import load, to_arrays, STATES
from bioreactor.model import simulate, predict_at_obs, NAMES, P0
from bioreactor.fit import fit

df, tmpl, train_ids, test_ids = load(ROOT / "data")
splits = pd.read_csv(ROOT / "outputs" / "splits.csv")
dev_ids = splits.query("role == 'dev'").batch_id.tolist()
hold_ids = splits.query("role == 'holdout'").batch_id.tolist()
fold_of = splits.query("role == 'dev'").set_index("batch_id").fold.to_dict()

# residual scale per state = spread of the measurements (puts VCD and g/L on equal footing)
meas = df[df.batch_id.isin(dev_ids) & df.vcd.notna()]
SCALE = meas[STATES].std().values
print("dev:", len(dev_ids), " holdout:", len(hold_ids), " test:", len(test_ids))
print("residual scale:", dict(zip(STATES, SCALE.round(3))))

## Helpers: metrics, baseline, plots

In [ ]:
def metrics(arr, preds):
    """RMSE, NRMSE (= RMSE / mean of observed) and R2 per state over all samples of arr."""
    Y = np.vstack(arr.obs_y); Yh = np.vstack(preds)
    rmse = np.sqrt(np.mean((Yh - Y) ** 2, axis=0))
    r2 = 1 - np.sum((Yh - Y) ** 2, axis=0) / np.sum((Y - Y.mean(0)) ** 2, axis=0)
    return pd.DataFrame({"RMSE": rmse, "NRMSE": rmse / Y.mean(0), "R2": r2}, index=STATES).round(3)

def baseline_preds(train_ids_, arr):
    """Naive reference: median training curve per day (ignores day-0 values and feeds)."""
    m = df[df.batch_id.isin(train_ids_) & df.vcd.notna()].copy()
    m["day"] = m.elapsed_time.round().astype(int)
    med = m.groupby("day")[STATES].median()
    return [med.reindex(np.round(t).astype(int)).values for t in arr.obs_t]

def plot_fit(p, arr, ids=None, title=""):
    ids = ids or arr.ids[:4]
    traj = simulate(p, arr)
    fig, ax = plt.subplots(len(ids), 4, figsize=(16, 2.8 * len(ids)), squeeze=False)
    for r, b in enumerate(ids):
        j = arr.ids.index(b)
        for c, v in enumerate(STATES):
            a = ax[r, c]
            a.plot(arr.t, traj[:, j, c], "-", c="tab:blue", label="model")
            a.plot(arr.obs_t[j], arr.obs_y[j][:, c], "o", c="k", ms=4, label="data")
            a.set_title(f"{b}: {v}", fontsize=9)
    ax[0, 0].legend(fontsize=8); fig.suptitle(title); fig.tight_layout(); plt.show()

def parity(arr, preds, title=""):
    Y = np.vstack(arr.obs_y); Yh = np.vstack(preds)
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
    for c, v in enumerate(STATES):
        ax[c].scatter(Y[:, c], Yh[:, c], s=8, alpha=0.6)
        lo, hi = min(Y[:, c].min(), Yh[:, c].min()), max(Y[:, c].max(), Yh[:, c].max())
        ax[c].plot([lo, hi], [lo, hi], "k--", lw=1); ax[c].set_xlabel("observed"); ax[c].set_ylabel("predicted"); ax[c].set_title(v)
    fig.suptitle(title); fig.tight_layout(); plt.show()

## 1. Sanity check of the simulator (no fitting)
Initial-guess parameters on a few dev batches. The curves will be off — the point is that the simulation runs, feeds produce jumps, and nothing goes negative / explodes.

In [ ]:
arr_dev = to_arrays(df, dev_ids)
t0 = time.time(); traj = simulate(P0, arr_dev); print(f"one simulation of {len(dev_ids)} batches: {time.time()-t0:.2f} s")
print("any NaN:", np.isnan(traj).any(), " min value:", traj.min().round(3))
plot_fit(P0, arr_dev, title="initial guess (not fitted)")

## 2. Optional: check the Euler step size
Halve dt and compare — if the curves barely change, dt = 0.005 d is fine.

In [ ]:
arr_fine = to_arrays(df, dev_ids[:3], dt=0.0025)
arr_std = to_arrays(df, dev_ids[:3], dt=0.005)
a = simulate(P0, arr_std)[::1]; b = simulate(P0, arr_fine)[::2][: len(a)]
print("max relative difference per state:", (np.abs(a - b).max(axis=(0, 1)) / np.abs(a).max(axis=(0, 1))).round(4))

## 3. Quick fit on one CV fold
Train on 4 folds of the dev set, validate on the held-out fold. `MAX_NFEV` limits the number of model evaluations — start small, increase if the fit is still improving.

In [ ]:
FOLD = 0
MAX_NFEV = 40

tr_ids = [b for b in dev_ids if fold_of[b] != FOLD]
va_ids = [b for b in dev_ids if fold_of[b] == FOLD]
arr_tr, arr_va = to_arrays(df, tr_ids), to_arrays(df, va_ids)

t0 = time.time()
p_fold, sol = fit(arr_tr, SCALE, max_nfev=MAX_NFEV, verbose=2)
print(f"fit time {time.time()-t0:.0f} s, cost {sol.cost:.2f}, status: {sol.message}")
pd.Series(p_fold, index=NAMES).round(4)

In [ ]:
pred_va = predict_at_obs(p_fold, arr_va)
print(f"fold {FOLD} — mechanistic model on unseen batches:"); display(metrics(arr_va, pred_va))
print("baseline (median training curve):"); display(metrics(arr_va, baseline_preds(tr_ids, arr_va)))
plot_fit(p_fold, arr_va, title=f"fold {FOLD}: validation batches (unseen)")
parity(arr_va, pred_va, title=f"fold {FOLD} validation")

In [ ]:
# training fit, to compare with validation (big gap = overfitting, both bad = model structure)
display(metrics(arr_tr, predict_at_obs(p_fold, arr_tr)))

## 4. Full 5-fold cross-validation
Run when section 3 looks reasonable. Warm-starts each fold from `p_fold` to save time.

In [ ]:
RUN_CV = False
if RUN_CV:
    cv_rows, cv_params = [], {}
    for f in range(5):
        tr_ = [b for b in dev_ids if fold_of[b] != f]; va_ = [b for b in dev_ids if fold_of[b] == f]
        a_tr, a_va = to_arrays(df, tr_), to_arrays(df, va_)
        p_f, _ = fit(a_tr, SCALE, p_init=p_fold, max_nfev=MAX_NFEV)
        cv_params[f] = p_f
        m_mech = metrics(a_va, predict_at_obs(p_f, a_va)).assign(model="mechanistic", fold=f)
        m_base = metrics(a_va, baseline_preds(tr_, a_va)).assign(model="baseline", fold=f)
        cv_rows += [m_mech, m_base]; print(f"fold {f} done")
    cv = pd.concat(cv_rows).rename_axis("state").reset_index()
    display(cv.groupby(["model", "state"])[["RMSE", "NRMSE", "R2"]].agg(["mean", "std"]).round(3))
    display(pd.DataFrame(cv_params, index=NAMES).round(4))  # parameter stability across folds

## 5. Hold-out check — run ONCE, at the end
Fit on all 39 dev batches, evaluate on the 10 hold-out batches. This is the honest estimate of performance on the hidden test set.

In [ ]:
RUN_HOLDOUT = False
if RUN_HOLDOUT:
    arr_all_dev, arr_hold = to_arrays(df, dev_ids), to_arrays(df, hold_ids)
    p_dev, _ = fit(arr_all_dev, SCALE, p_init=p_fold, max_nfev=MAX_NFEV)
    pred_h = predict_at_obs(p_dev, arr_hold)
    display(metrics(arr_hold, pred_h)); display(metrics(arr_hold, baseline_preds(dev_ids, arr_hold)))
    plot_fit(p_dev, arr_hold, ids=hold_ids[:5], title="hold-out batches")

## 6. Final fit on all 49 batches → submission
Predicts the 12 test batches at the template time points and writes `outputs/submission_mechanistic.csv`.

In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    arr_all = to_arrays(df, train_ids)
    p_final, _ = fit(arr_all, SCALE, p_init=p_fold, max_nfev=MAX_NFEV)
    arr_test = to_arrays(df, test_ids, sample_mask="predict")
    preds = predict_at_obs(p_final, arr_test)
    rows = [dict(batch_id=b, elapsed_time=t, **dict(zip(STATES, y)))
            for b, ts, ys in zip(arr_test.ids, arr_test.obs_t, preds) for t, y in zip(ts, ys)]
    pred_df = pd.DataFrame(rows)
    sub = tmpl[["batch_id", "elapsed_time"]].merge(pred_df, on=["batch_id", "elapsed_time"], how="left")
    assert len(sub) == len(tmpl) and sub[STATES].notna().all().all(), "template rows not all matched"
    sub.to_csv(ROOT / "outputs" / "submission_mechanistic.csv", index=False)
    pd.Series(p_final, index=NAMES).to_csv(ROOT / "outputs" / "params_mechanistic.csv")
    plot_fit(p_final, arr_test, ids=test_ids[:4], title="test batches (prediction only)")
    sub.head()